# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dishaa34/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

I rank content pages based on three historical signals:

1. Content that has not been updated for a long time.
2. Content with high search volume.
3. Content with a low click-through rate (CTR).

Pages with higher scores are considered stronger candidates for a content refresh.

### Reason Codes

- **STALE_CONTENT** – The content has not been updated recently.
- **LOW_CTR** – The page has a low click-through rate despite receiving impressions.
- **HIGH_SEARCH_VOLUME** – The page has high search demand and could benefit from optimization.

### Action Label

- **Refresh Content**

In [2]:
import pandas as pd

df = pd.read_csv("/content_refresh_anonymized.csv")

print(df[[
    "days_since_last_update",
    "search_volume",
    "ctr"
]].describe())

       days_since_last_update  search_volume           ctr
count            28059.000000   25749.000000  28059.000000
mean                46.033536     157.671366      0.512848
std                 42.032879    1504.168634      3.314642
min                  1.000000       0.000000      0.000000
25%                 20.000000       0.000000      0.000000
50%                 20.000000      10.000000      0.070000
75%                104.000000      20.000000      0.280000
max                373.000000   74000.000000    100.000000


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Build the Ranked Queue

The baseline score combines three historical signals:

- Content age
- Search volume
- Low CTR

Only historical information is used. No future or label-derived fields are included.

In [3]:
import os

# Normalize signals
age_score = df["days_since_last_update"] / df["days_since_last_update"].max()

volume_score = df["search_volume"] / df["search_volume"].max()

ctr_score = 1 - (df["ctr"] / df["ctr"].max())

# Final baseline score
df["baseline_score"] = (
    0.4 * age_score +
    0.3 * volume_score +
    0.3 * ctr_score
)

# Reason codes
def reason(row):

    if row["days_since_last_update"] >= 180:
        return "STALE_CONTENT"

    elif row["ctr"] < 0.03:
        return "LOW_CTR"

    else:
        return "HIGH_SEARCH_VOLUME"

df["reason_code"] = df.apply(reason, axis=1)

# Action
df["action"] = "Refresh Content"

# Rank
ranked = df.sort_values(
    "baseline_score",
    ascending=False
)

# Save CSV
os.makedirs("work/outputs", exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")

ranked.head(10)

CSV written successfully.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
12140,content_ef99c4abd9ab,client_3fdba35f04,74000.0,0.08,LOW,0.34,keyword article,informational,NaN,NaN,...,0.0,5.00,0.0,good,page_3_5,stable,3.6,0.711438,HIGH_SEARCH_VOLUME,Refresh Content
26242,content_55a5b1c46474,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,low,page_1,down,-88.5,0.700000,STALE_CONTENT,Refresh Content
18440,content_8d56efff1e71,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,50.00,0.0,low,page_3_5,new,NaN,0.698928,STALE_CONTENT,Refresh Content
24216,content_1b4ec72dafd4,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,50.00,0.0,low,page_1,down,-100.0,0.698928,STALE_CONTENT,Refresh Content
6972,content_bf67a444faef,client_3fdba35f04,60500.0,0.11,LOW,0.50,keyword article,informational,NaN,NaN,...,0.0,13.33,0.0,good,page_3_5,down,-20.0,0.656798,LOW_CTR,Refresh Content
18701,content_deb54e9e19cd,client_3fdba35f04,60500.0,0.13,LOW,0.56,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,good,page_3_5,stable,-2.5,0.656798,LOW_CTR,Refresh Content
17907,content_5ec29ae79c60,client_3fdba35f04,60500.0,0.13,LOW,0.76,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,moderate,page_3_5,up,73.0,0.656798,LOW_CTR,Refresh Content
21984,content_02b0d6e30129,client_19581e27de,110.0,0.40,MEDIUM,0.59,keyword article,transactional,NaN,NaN,...,0.0,0.00,0.0,low,page_1,down,-95.6,0.636103,STALE_CONTENT,Refresh Content
7509,content_7a888d3d99c8,client_19581e27de,90.0,0.46,MEDIUM,0.72,keyword article,transactional,NaN,NaN,...,0.0,0.00,0.0,low,deep,down,-100.0,0.636022,STALE_CONTENT,Refresh Content
15790,content_6476d1d8c050,client_19581e27de,10.0,0.54,MEDIUM,0.85,keyword article,transactional,NaN,NaN,...,0.0,0.00,0.0,moderate,deep,up,31.9,0.635697,STALE_CONTENT,Refresh Content


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

Each recommendation should be interpreted as decision support rather than a guaranteed improvement opportunity.

For every page, the recommendation is based on historical signals only.

In [4]:
top20 = ranked.head(20)[[
    "content_id",
    "baseline_score",
    "reason_code",
    "action"
]]

top20["confidence_note"] = (
    "Medium confidence based on historical metrics."
)

top20["what_would_make_it_wrong"] = (
    "Recent updates, seasonality, or missing business context."
)

top20

,content_id,baseline_score,reason_code,action,confidence_note,what_would_make_it_wrong
12140,content_ef99c4abd9ab,0.711438,HIGH_SEARCH_VOLUME,Refresh Content,Medium confidence based on historical metrics.,"Recent updates, seasonality, or missing busine..."
26242,content_55a5b1c46474,0.700000,STALE_CONTENT,Refresh Content,Medium confidence based on historical metrics.,"Recent updates, seasonality, or missing busine..."
18440,content_8d56efff1e71,0.698928,STALE_CONTENT,Refresh Content,Medium confidence based on historical metrics.,"Recent updates, seasonality, or missing busine..."
24216,content_1b4ec72dafd4,0.698928,STALE_CONTENT,Refresh Content,Medium confidence based on historical metrics.,"Recent updates, seasonality, or missing busine..."
6972,content_bf67a444faef,0.656798,LOW_CTR,Refresh Content,Medium confidence based on historical metrics.,"Recent updates, seasonality, or missing busine..."
18701,content_deb54e9e19cd,0.656798,LOW_CTR,Refresh Content,Medium confidence based on historical metrics.,"Recent updates, seasonality, or missing busine..."
17907,content_5ec29ae79c60,0.656798,LOW_CTR,Refresh Content,Medium confidence based on historical metrics.,"Recent updates, seasonality, or missing busine..."
21984,content_02b0d6e30129,0.636103,STALE_CONTENT,Refresh Content,Medium confidence based on historical metrics.,"Recent updates, seasonality, or missing busine..."
7509,content_7a888d3d99c8,0.636022,STALE_CONTENT,Refresh Content,Medium confidence based on historical metrics.,"Recent updates, seasonality, or missing busine..."
15790,content_6476d1d8c050,0.635697,STALE_CONTENT,Refresh Content,Medium confidence based on historical metrics.,"Recent updates, seasonality, or missing busine..."


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages may rank highly because they are old but still perform well.

Seasonality or recent manual updates are not represented in this dataset.

### Leakage Check

The baseline rule does not use future outcomes or label-derived columns.

Columns such as `trend_pct` and `trend_direction` were intentionally excluded because they represent outcome information and could introduce target leakage.

In [5]:
excluded = [
    "trend_pct",
    "trend_direction"
]

print("Excluded columns:")
print(excluded)

print("\nNo future-window or label-derived columns were used in the baseline score.")

Excluded columns:
['trend_pct', 'trend_direction']

No future-window or label-derived columns were used in the baseline score.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.